# SME Credit Risk Detection & Underwriting Model
### Machine Learning Pipeline for Risk Scoring & Classification

This notebook trains and evaluates the machine learning model used in the **SME Credit Risk Assessment System**.
It outputs the trained artifacts (`sme_risk_model.pkl`, `feature_columns.pkl`, `districts.pkl`, `location_types.pkl`, `sectors.pkl`) directly compatible with the Flask deployment (`app.py`).

🔗 **Google Colab Link**: [Open in Google Colab](https://colab.research.google.com/drive/15VWt3_JMnwcNj3xKY0xRSH4c-5-Jr0DC)

## 1. Import Libraries & Dependencies

In [ ]:
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Libraries loaded successfully!")

## 2. Dataset Preparation / Generation
*(If you have your own CSV dataset, upload it and use `df = pd.read_csv('your_dataset.csv')`)*

In [ ]:
np.random.seed(42)
N_SAMPLES = 5000

districts_list = ["Dhaka", "Chattogram", "Sylhet", "Rajshahi", "Khulna", "Barishal", "Rangpur", "Mymensingh"]
location_types_list = ["Urban", "Semi-Urban", "Rural"]
sectors_list = ["Retail Trade", "Wholesale Trade", "Manufacturing", "Agro-Processing", "IT & Software", "Hospitality & Services"]

business_age = np.random.exponential(scale=5, size=N_SAMPLES).clip(0.5, 30).round(1)
owner_exp = (business_age + np.random.normal(3, 2, N_SAMPLES)).clip(1, 40).round(1)
banking_years = np.minimum(business_age, np.random.exponential(scale=4, size=N_SAMPLES)).clip(0.5, 25).round(1)

annual_revenue = np.random.lognormal(mean=14.5, sigma=0.8, size=N_SAMPLES).round(-3)
loan_amount = (annual_revenue * np.random.uniform(0.15, 0.75, N_SAMPLES)).round(-3)
debt_to_asset_ratio = np.random.beta(2, 5, N_SAMPLES).clip(0.05, 0.95).round(2)
num_employees = np.random.poisson(lam=12, size=N_SAMPLES).clip(1, 150)

has_collat = np.random.choice([0, 1], size=N_SAMPLES, p=[0.25, 0.75])
collateral_value = (has_collat * loan_amount * np.random.uniform(0.5, 2.0, N_SAMPLES)).round(-3)
past_defaults = np.random.choice([0, 1, 2, 3], size=N_SAMPLES, p=[0.75, 0.15, 0.07, 0.03])

districts = np.random.choice(districts_list, size=N_SAMPLES)
location_types = np.random.choice(location_types_list, size=N_SAMPLES, p=[0.55, 0.30, 0.15])
sectors = np.random.choice(sectors_list, size=N_SAMPLES)

df = pd.DataFrame({
    "business_age_years": business_age,
    "owner_experience_years": owner_exp,
    "banking_years": banking_years,
    "annual_revenue": annual_revenue,
    "loan_amount": loan_amount,
    "debt_to_asset_ratio": debt_to_asset_ratio,
    "num_employees": num_employees,
    "collateral_value": collateral_value,
    "past_defaults": past_defaults,
    "district": districts,
    "location_type": location_types,
    "sector": sectors
})

def assign_risk_label(row):
    score = 0
    score += row['past_defaults'] * 40
    collat_ratio = (row['collateral_value'] / row['loan_amount']) if row['loan_amount'] > 0 else 0
    if collat_ratio < 0.5: score += 25
    elif collat_ratio < 1.0: score += 10
    if row['debt_to_asset_ratio'] > 0.65: score += 25
    elif row['debt_to_asset_ratio'] > 0.45: score += 10
    if row['business_age_years'] < 2: score += 15
    if (row['loan_amount'] / row['annual_revenue']) > 0.5: score += 15

    if score <= 20: return "LOW"
    elif score <= 50: return "MEDIUM"
    else: return "HIGH"

df['risk_level'] = df.apply(assign_risk_label, axis=1)
print("Dataset Shape:", df.shape)
print("\nRisk Distribution:\n", df['risk_level'].value_counts())
df.head()

## 3. Feature Engineering & One-Hot Encoding

In [ ]:
X_raw = df.drop(columns=['risk_level']).copy()
y = df['risk_level']

# Derived Financial Features
X_raw['loan_to_revenue'] = np.where(X_raw['annual_revenue'] > 0, X_raw['loan_amount'] / X_raw['annual_revenue'], 0.0)
X_raw['collateral_coverage'] = np.where(X_raw['loan_amount'] > 0, X_raw['collateral_value'] / X_raw['loan_amount'], 0.0)
X_raw['has_collateral'] = (X_raw['collateral_value'] > 0).astype(int)

# One-Hot Encoding
X = pd.get_dummies(X_raw, columns=['district', 'location_type', 'sector'], dtype=int)
feature_columns = list(X.columns)

print(f"Total Features: {len(feature_columns)}")
print("Features:", feature_columns)

## 4. Train Random Forest Model & Evaluate

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred, labels=["LOW", "MEDIUM", "HIGH"]), 
            annot=True, fmt='d', cmap='Blues', 
            xticklabels=["LOW", "MEDIUM", "HIGH"], 
            yticklabels=["LOW", "MEDIUM", "HIGH"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 5. Export Model & Metadata (.pkl) for Web Deployment

In [ ]:
unique_districts = sorted(df['district'].unique().tolist())
unique_location_types = sorted(df['location_type'].unique().tolist())
unique_sectors = sorted(df['sector'].unique().tolist())

joblib.dump(model, "sme_risk_model.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")
joblib.dump(unique_districts, "districts.pkl")
joblib.dump(unique_location_types, "location_types.pkl")
joblib.dump(unique_sectors, "sectors.pkl")

print("All 5 model artifacts exported successfully!")

# If running in Google Colab, trigger automatic download:
try:
    from google.colab import files
    for fname in ["sme_risk_model.pkl", "feature_columns.pkl", "districts.pkl", "location_types.pkl", "sectors.pkl"]:
        files.download(fname)
except ImportError:
    print("Running locally: artifacts are saved in current directory.")